In [9]:
import os

In [21]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\emotion_detection'

In [11]:
os.chdir("../")

In [12]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\emotion_detection'

In [13]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DatainjectionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [14]:
from emotion_detection.constant import *
from emotion_detection.utils.common import read_yaml, create_directories

In [15]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_injection_config(self) -> DatainjectionConfig:
        config = self.config.data_injection

        create_directories([config.root_dir])

        data_injection_config = DatainjectionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_injection_config

In [16]:
import os
import urllib.request as request
import zipfile
from emotion_detection.logging import logger
from emotion_detection.utils.common import get_size

In [17]:
# 5 Conponents

class DataInjection:
    def __init__(self, config: DatainjectionConfig):
        self.config = config


    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded with following info \n{headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zipfile(self):
        """
        zip_file_path: str
        Extracts the zip file in dir
        None returns
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [23]:
# 6 Pipeline

try:
    config = configurationManager()
    data_injection_config = config.get_data_injection_config()
    data_injection = DataInjection(config = data_injection_config)
    data_injection.download_files()
    data_injection.extract_zipfile()

except Exception as e:
    raise e

[2026-06-21 23:47:41,947: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-06-21 23:47:41,950: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-06-21 23:47:41,951: INFO: common: created directory at artifacts]
[2026-06-21 23:47:41,953: INFO: common: created directory at artifacts/data_injection]
[2026-06-21 23:47:41,954: INFO: 4053066924: file already exist of size: ~ 39212 KB]
